# River Video Analysis – My Own River Video

This notebook records the actual Jupyter/Python workflow used for my own river video **`IMG_9373 (1).MOV`**.

**Important:** The Ngwerere example video was used only for learning/testing PyORC and is not included as my project dataset or final result.

The optical-flow results in this notebook are **image-space measurements in pixels**. They are not calibrated river velocity in m/s, PIV velocity, or discharge.

## 1. Import libraries and check the environment

This was used to verify the Python executable, PyORC version, and OpenCV version.

In [ ]:
import sys
import inspect
from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt
import cv2
import pyorc

print("Python:", sys.executable)
print("pyorc:", pyorc.__version__)
print("OpenCV:", cv2.__version__)

## 2. Locate the project and own river video

The repository uses a project-relative path so the notebook can be moved with the project.

In [ ]:
# Find the project root whether Jupyter starts in the project root or notebooks folder
PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / "data" / "raw").exists() and (PROJECT_ROOT.parent / "data" / "raw").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent

video_file = PROJECT_ROOT / "data" / "raw" / "IMG_9373 (1).MOV"

print("Project folder:", PROJECT_ROOT)
print("Video exists:", video_file.exists())
print("Video path:", video_file)

## 3. Load the own video with PyORC

The video loaded successfully. PyORC reported a warning that the final frame could not be read and adapted the end frame from 194 to 193.

In [ ]:
video = pyorc.Video(str(video_file))

print("Video loaded successfully!")
print("Width:", video.width)
print("Height:", video.height)
print("FPS:", video.fps)
print("Start frame:", video.start_frame)
print("End frame:", video.end_frame)

## 4. Display a frame from the own river video

Frame 0 and frame 30 were loaded for visual inspection. Frame 30 was displayed.

In [ ]:
frame0 = video.get_frame(0, method="rgb")
frame30 = video.get_frame(30, method="rgb")

print("Frame 0 shape:", frame0.shape)
print("Frame 30 shape:", frame30.shape)

plt.figure(figsize=(12, 7))
plt.imshow(frame30)
plt.axis("off")
plt.title("Own River Video - Frame 30")
plt.show()

## 5. Basic frame-difference test

A simple grayscale pixel difference was used to check whether the video contains visible changes between frames 0 and 30.

In [ ]:
gray0 = cv2.cvtColor(frame0, cv2.COLOR_RGB2GRAY)
gray30 = cv2.cvtColor(frame30, cv2.COLOR_RGB2GRAY)

diff = cv2.absdiff(gray0, gray30)

print("Difference min:", diff.min())
print("Difference max:", diff.max())
print("Difference average:", diff.mean())

## 6. Display the movement/difference map

In [ ]:
plt.figure(figsize=(12, 7))
plt.imshow(diff, cmap="gray")
plt.axis("off")
plt.title("Frame Difference: 0 to 30")
plt.colorbar(label="Absolute pixel difference")
plt.show()

## 7. Test a short video range

A shorter PyORC video object was created using frames 0–60.

In [ ]:
video_test = pyorc.Video(str(video_file), start_frame=0, end_frame=60)
print("Short video loaded successfully!")
print("Start frame:", video_test.start_frame)
print("End frame:", video_test.end_frame)

## 8. PyORC frame extraction test without camera calibration

This was tested before camera calibration. It produced an expected error because `get_frames()` requires a camera configuration.

The failed command is preserved below as a comment so the notebook can still run from top to bottom.

In [ ]:
# Historical test that failed because no camera configuration was set:
# frames = video_test.get_frames()
# Error: AssertionError: No camera configuration is set, add it to the video using the .camera_config method

## 9. Load individual frames for optical-flow analysis

Frames 0, 15, 30, and 45 were checked. The video frames were RGB images with shape `(1080, 1920, 3)`.

In [ ]:
frame_numbers = [0, 15, 30, 45]
frames = {}

for n in frame_numbers:
    frames[n] = video.get_frame(n, method="rgb")
    print(f"Frame {n}:", frames[n].shape)

## 10. Farneback optical flow: frame 0 to frame 30

OpenCV Farneback optical flow was used to estimate image movement between two frames. The output is measured in pixels.

In [ ]:
gray0 = cv2.cvtColor(frames[0], cv2.COLOR_RGB2GRAY)
gray30 = cv2.cvtColor(frames[30], cv2.COLOR_RGB2GRAY)

flow = cv2.calcOpticalFlowFarneback(
    gray0, gray30, None,
    0.5, 3, 15, 3, 5, 1.2, 0
)

flow_speed = np.sqrt(flow[..., 0]**2 + flow[..., 1]**2)

print("Average movement:", flow_speed.mean(), "pixels")
print("Maximum movement:", flow_speed.max(), "pixels")

## 11. Quiver plot of optical flow

The arrows show the estimated image movement direction and magnitude.

In [ ]:
step = 60
y, x = np.mgrid[0:flow.shape[0]:step, 0:flow.shape[1]:step]
u = flow[::step, ::step, 0]
v = flow[::step, ::step, 1]

plt.figure(figsize=(12, 7))
plt.imshow(frames[30])
plt.quiver(x, y, u, v, scale=300)
plt.axis("off")
plt.title("Optical Flow: Frame 0 to Frame 30")
plt.show()

## 12. Water-region optical flow

The lower part of the image was used as a rough water region. This was an image-space test only; it was not a calibrated river mask.

In [ ]:
water_flow = flow[450:1080, :, :]
water_speed = np.sqrt(water_flow[..., 0]**2 + water_flow[..., 1]**2)

print("Water ROI average movement:", water_speed.mean(), "pixels")
print("Water ROI maximum movement:", water_speed.max(), "pixels")

## 13. Improved water ROI

A more focused region was tested using rows 500–900 and columns 200–1800.

In [ ]:
roi_flow = flow[500:900, 200:1800, :]
roi_speed = np.sqrt(roi_flow[..., 0]**2 + roi_flow[..., 1]**2)

print("Improved water ROI average movement:", roi_speed.mean(), "pixels")
print("Improved water ROI maximum movement:", roi_speed.max(), "pixels")

## 14. Strong-movement threshold

A threshold of 2.0 pixels was used to highlight stronger image movement in the improved water ROI.

In [ ]:
threshold = 2.0
strong_movement = roi_speed > threshold

plt.figure(figsize=(12, 7))
plt.imshow(strong_movement, cmap="gray")
plt.axis("off")
plt.title("Strong Movement in Water ROI (> 2 pixels)")
plt.show()

## 15. Improved water optical-flow quiver plot

In [ ]:
step = 40
roi = flow[500:900, 200:1800, :]
y, x = np.mgrid[0:roi.shape[0]:step, 0:roi.shape[1]:step]
u = roi[::step, ::step, 0]
v = roi[::step, ::step, 1]

plt.figure(figsize=(12, 6))
plt.imshow(frames[30][500:900, 200:1800])
plt.quiver(x, y, u, v, scale=150)
plt.axis("off")
plt.title("Water ROI Optical Flow: Frame 0 to Frame 30")
plt.show()

## 16. Consistency test across several frame pairs

Five 30-frame intervals were tested: 0→30, 30→60, 60→90, 90→120, and 120→150.

In [ ]:
pairs = [(0, 30), (30, 60), (60, 90), (90, 120), (120, 150)]
results = []

for start, end in pairs:
    f1 = video.get_frame(start, method="rgb")
    f2 = video.get_frame(end, method="rgb")
    g1 = cv2.cvtColor(f1, cv2.COLOR_RGB2GRAY)
    g2 = cv2.cvtColor(f2, cv2.COLOR_RGB2GRAY)
    pair_flow = cv2.calcOpticalFlowFarneback(
        g1, g2, None,
        0.5, 3, 15, 3, 5, 1.2, 0
    )
    pair_speed = np.sqrt(pair_flow[..., 0]**2 + pair_flow[..., 1]**2)
    avg_movement = pair_speed.mean()
    max_movement = pair_speed.max()
    results.append((start, end, avg_movement, max_movement, pair_flow))
    print(f"{start}->{end}: average = {avg_movement:.6f} pixels, max = {max_movement:.6f} pixels")

### Recorded consistency-test values

These are the values obtained during the previous test run:

In [ ]:
# Recorded values from the previous run (for comparison/documentation).
recorded_results = [
    (0, 30, 5.669003, 53.404606),
    (30, 60, 8.373936, 65.832320),
    (60, 90, 6.983921, 84.969185),
    (90, 120, 4.867338, 58.564100),
    (120, 150, 8.586046, 72.188240),
]

for start, end, avg_movement, max_movement in recorded_results:
    print(f"{start}->{end}: average = {avg_movement:.6f} pixels, max = {max_movement:.6f} pixels")

## 17. Overall average image movement

In [ ]:
overall_average = np.mean([r[2] for r in recorded_results])
print("Overall average movement:", overall_average, "pixels")

## 18. Average X/Y movement components

The X and Y components describe image-space direction. Positive X means rightward image movement; negative Y means upward image movement.

In [ ]:
component_results = []

for start, end in pairs:
    f1 = video.get_frame(start, method="rgb")
    f2 = video.get_frame(end, method="rgb")
    g1 = cv2.cvtColor(f1, cv2.COLOR_RGB2GRAY)
    g2 = cv2.cvtColor(f2, cv2.COLOR_RGB2GRAY)
    pair_flow = cv2.calcOpticalFlowFarneback(
        g1, g2, None,
        0.5, 3, 15, 3, 5, 1.2, 0
    )
    component_results.append((pair_flow[..., 0].mean(), pair_flow[..., 1].mean()))

avg_x = np.mean([r[0] for r in component_results])
avg_y = np.mean([r[1] for r in component_results])

print("Average X movement:", avg_x, "pixels")
print("Average Y movement:", avg_y, "pixels")

### Recorded X/Y values from the previous run

- Average X: **+2.7479858 pixels**
- Average Y: **-0.7804416 pixels**
- Overall average movement: **6.8960485 pixels**

These values are image-space optical-flow measurements only.

## 19. Calibrated PyORC stage — TO BE COMPLETED

The own river video does not yet have verified camera calibration/GCP information in this project. Therefore, true projected coordinates, calibrated surface velocity in m/s, bulk velocity, and discharge should not be claimed yet.

When real camera configuration becomes available, inspect the installed API before running version-sensitive operations.

In [ ]:
# Example structure for the future calibrated stage.
# Do not uncomment until real camera configuration/GCP information is available.
#
# camera_config_path = PROJECT_ROOT / "data" / "config" / "camera_config.json"
# cam_config = pyorc.load_camera_config(str(camera_config_path))
# video_calibrated = pyorc.Video(str(video_file), camera_config=cam_config)
# frames_calibrated = video_calibrated.get_frames(method="grayscale")
# frames_obj = pyorc.Frames(frames_calibrated)
# print(inspect.signature(frames_obj.project))
# print(inspect.signature(frames_obj.get_piv))
# projected = frames_obj.project(method="numpy", resolution=0.01)
# piv = frames_obj.get_piv(
#     window_size=(25, 25),
#     overlap=(12, 12),
#     search_area_size=(25, 25),
#     engine="numba"
# )

## 20. Future velocity and quality filtering — TO BE COMPLETED

In [ ]:
# Future calibrated PIV processing:
# velocity = np.sqrt(piv.v_x**2 + piv.v_y**2)
# quality_mask = (piv.s2n >= 3) & (piv.corr >= 0.2)
# velocity_filtered = velocity.where(quality_mask)

## 21. Transect, surface velocity, bulk velocity, and discharge — IN PROGRESS

The actual cross-section and elevation information are still required. Do not insert invented x/y/z coordinates or discharge values. The exact installed PyORC API should be inspected before using transect, surface velocity, bulk velocity, or discharge methods.